# Agent Architecture & Memory — a minimal agent from scratch

> Wiki: [Agent Architecture & Memory](https://ml-viz-ruby.vercel.app/wiki/agent-architecture-and-memory)
> · Copy to Drive to run and edit.

A functioning agent is a **loop** over four elements — an input interface, a reasoning
core, tools, and memory — and it runs the cycle **Perceive → Recall → Reason → Act →
Store** on every turn. To *see* the machinery, we build one with **no LLM and no
framework**: the "reasoning core" is a tiny deterministic policy, so the whole thing runs
offline and identically every time.

The point is not the policy — it's **where memory is read and written**, and how the three
memory layers (short-term, long-term, external) each play a distinct role. We'll build the
pieces incrementally, run a multi-turn support conversation, and then hand you the scaffold
to extend it.

## 1 · The three memory layers

Agent memory differs along **scope, purpose, and persistence**:

| Layer | Scope | Holds | Here |
|---|---|---|---|
| **Short-term** | one session | recent turns | a list, reset per session |
| **Long-term** | across sessions | preferences, ticket history | a dict that survives resets |
| **External** | outside the agent | reference docs | a keyword-scored knowledge base |

We model each as a plain Python object so the reads and writes are explicit.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Memory:
    """Three layers, deliberately separated so every access is visible."""
    short_term: list = field(default_factory=list)          # (role, text) this session
    long_term: dict = field(default_factory=dict)           # user_id -> durable facts
    external: dict = field(default_factory=dict)            # doc_id  -> text (knowledge base)

    def new_session(self):
        """Short-term is fleeting: cleared each session. Long-term & external persist."""
        self.short_term = []

mem = Memory()
mem.external = {
    "billing_faq": "To change your plan, open Settings > Billing and pick a new tier. "
                   "Refunds are prorated automatically.",
    "login_faq":   "If you are locked out, use 'Forgot password' to reset. "
                   "Accounts lock after 5 failed attempts for 15 minutes.",
    "escalation":  "Frustrated customers should be escalated to a human agent after "
                   "two negative-sentiment turns.",
}
mem.long_term["u_42"] = {
    "name": "Ada",
    "open_tickets": [{"id": "T-100", "summary": "billing overcharge", "status": "open",
                      "opened": "yesterday"}],
    "sentiment_history": [],
}
print("external docs:", list(mem.external))
print("known user   :", mem.long_term["u_42"]["name"])

## 2 · Perceive — turn raw input into signals

Perception sits at the **start** of the loop: raw input becomes structured signals the
core can use. Real agents run speech-to-text, OCR, or JSON parsing here; ours does light
intent + sentiment extraction so the rest of the loop has something to condition on.

In [ ]:
NEG_WORDS = {"angry", "furious", "ridiculous", "unacceptable", "again", "still", "worst"}

def perceive(raw_text: str) -> dict:
    """Raw string -> signal dict (intent, sentiment, keywords)."""
    low = raw_text.lower()
    if any(w in low for w in ("bill", "charge", "refund", "plan", "overcharge")):
        intent = "billing"
    elif any(w in low for w in ("login", "log in", "locked", "password")):
        intent = "login"
    else:
        intent = "general"
    sentiment = "negative" if any(w in low for w in NEG_WORDS) else "neutral"
    keywords = [w.strip("?.!,") for w in low.split()]
    return {"text": raw_text, "intent": intent, "sentiment": sentiment, "keywords": keywords}

print(perceive("I'm still being overcharged on my bill, this is unacceptable!"))

## 3 · Recall — retrieve memory *before* reasoning

Retrieval happens first. We read **all three** layers: recent turns (short-term), the
user's durable facts and open tickets (long-term), and the most relevant knowledge-base
snippet (external). The external read is a stand-in for a **semantic search** over a
vector DB — here, a simple keyword-overlap score picks the best doc rather than loading
the whole base.

In [ ]:
def semantic_search(query_keywords, external, k=1):
    """Toy stand-in for a vector-DB similarity search: score docs by keyword overlap."""
    scored = []
    for doc_id, text in external.items():
        overlap = sum(1 for w in query_keywords if w and w in text.lower())
        scored.append((overlap, doc_id, text))
    scored.sort(reverse=True)
    return [(doc_id, text) for score, doc_id, text in scored[:k] if score > 0]

def recall(signal: dict, mem: Memory, user_id: str) -> dict:
    """Read short-term, long-term, and external memory into one context bundle."""
    return {
        "recent_turns": list(mem.short_term),                 # short-term
        "user": mem.long_term.get(user_id, {}),               # long-term
        "docs": semantic_search(signal["keywords"], mem.external),  # external
    }

mem.new_session()
sig = perceive("I'm still being overcharged, ticket from yesterday.")
ctx = recall(sig, mem, "u_42")
print("recalled user  :", ctx["user"]["name"], "| open:", ctx["user"]["open_tickets"])
print("recalled doc   :", ctx["docs"][0][0] if ctx["docs"] else None)

## 4 · Reason & plan, then Act

The core, conditioned on the recalled context, decides **what to do**: answer from a doc,
reference the open ticket, or escalate. `act` then executes the chosen action. Because we
recalled the ticket and the sentiment history, the agent can avoid re-asking and can
escalate when frustration repeats — the whole reason memory matters.

In [ ]:
def reason(signal: dict, ctx: dict) -> dict:
    """Pick an action from signal + recalled context. Returns a plan dict."""
    user = ctx["user"]
    # Count negative turns including this one -> escalate on the second.
    neg_history = user.get("sentiment_history", [])
    neg_count = neg_history.count("negative") + (1 if signal["sentiment"] == "negative" else 0)
    if neg_count >= 2:
        return {"action": "escalate", "reason": "repeated dissatisfaction"}
    if ctx["docs"]:
        return {"action": "answer", "doc": ctx["docs"][0][0], "text": ctx["docs"][0][1]}
    return {"action": "clarify"}

def act(plan: dict, ctx: dict) -> str:
    user = ctx["user"]
    ticket = (user.get("open_tickets") or [None])[0]
    if plan["action"] == "escalate":
        return "I hear your frustration \u2014 connecting you to a human agent now."
    if plan["action"] == "answer":
        ref = f" I can see your open ticket {ticket['id']} ({ticket['summary']})." if ticket else ""
        return f"{plan['text']}{ref}"
    return "Could you tell me a bit more so I can help?"

print(act(reason(sig, ctx), ctx))

## 5 · Store — write new memory *after* acting

After acting, we persist what happened: the turn goes into **short-term** memory, and
durable signals (here, the sentiment) go into **long-term** memory so the *next* session
can recall them. This closing write is what lets experience carry forward.

In [ ]:
def store(signal: dict, answer: str, mem: Memory, user_id: str):
    mem.short_term.append(("user", signal["text"]))   # short-term: the turn
    mem.short_term.append(("agent", answer))
    mem.long_term.setdefault(user_id, {}).setdefault("sentiment_history", []) \
        .append(signal["sentiment"])                  # long-term: durable signal

# One full turn = Perceive -> Recall -> Reason -> Act -> Store.
def agent_turn(raw_text: str, mem: Memory, user_id: str) -> str:
    signal = perceive(raw_text)
    ctx    = recall(signal, mem, user_id)
    plan   = reason(signal, ctx)
    answer = act(plan, ctx)
    store(signal, answer, mem, user_id)
    return answer

print("one turn:", agent_turn("How do I change my plan?", mem, "u_42"))

## 6 · The loop across a session

Run several turns. Watch two things: (1) the agent references the **long-term** ticket
without the user restating it, and (2) after the **second** negative turn it **escalates**
— because Store wrote the first negative sentiment to long-term memory and Recall read it
back.

In [ ]:
mem.new_session()   # fresh short-term; long-term (ticket, name) survives
mem.long_term["u_42"]["sentiment_history"] = []   # isolate this demo session's sentiment
conversation = [
    "Hi, I think I was overcharged on my bill.",
    "This is unacceptable, I want it fixed.",
    "I'm still furious, nothing has changed!",
]
for turn in conversation:
    print("USER :", turn)
    print("AGENT:", agent_turn(turn, mem, "u_42"))
    print()
print("sentiment_history (long-term):", mem.long_term["u_42"]["sentiment_history"])

### What to notice

- The agent **never re-asks** for the ticket — it was recalled from long-term memory.
- The **third** turn escalates: by then long-term memory holds two negative sentiments, so
  Recall → Reason sees the pattern. Remove the Store step and this never happens.
- Short-term memory (`mem.short_term`) grows within the session; calling `new_session()`
  clears it while long-term and external memory persist. That is the whole distinction
  between the layers, made concrete.

## 7 · Visualize — where memory is touched on each step

A quick figure to make the read/write rhythm stick: across the five loop steps, memory is
**read on Recall** and **written on Store** — nowhere else.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("dark_background")
steps = ["Perceive", "Recall", "Reason", "Act", "Store"]
reads  = [0, 3, 0, 0, 0]   # layers read on each step
writes = [0, 0, 0, 0, 2]   # layers written on each step

fig, ax = plt.subplots(figsize=(7, 3.2))
x = range(len(steps))
ax.bar([i - 0.18 for i in x], reads,  width=0.36, label="memory reads",  color="#a5b4fc")
ax.bar([i + 0.18 for i in x], writes, width=0.36, label="memory writes", color="#f43f5e")
ax.set_xticks(list(x)); ax.set_xticklabels(steps)
ax.set_ylabel("memory layers touched"); ax.set_yticks([0, 1, 2, 3])
ax.set_title("Memory is read before reasoning, written after acting")
ax.legend(frameon=False)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## 8 · ✏️ Your turn

Extend the agent. Each task has an `assert` that passes silently when you get it right,
and a solution in the `<details>` below.

**Task A — add a fourth intent.** Add a `"cancel"` intent to `perceive` (triggered by
words like *cancel*, *unsubscribe*) and a matching doc to `mem.external`, then confirm a
cancel query retrieves your new doc.

In [ ]:
# TODO(you): extend perceive() with a "cancel" intent, and add a cancellation doc.
# Then a cancel query should retrieve it via semantic_search.

def perceive_v2(raw_text: str) -> dict:
    sig = perceive(raw_text)
    # TODO(you): override sig["intent"] = "cancel" when the text is about cancelling
    return sig

# mem.external["cancel_faq"] = "..."   # TODO(you)

# --- check (uncomment once implemented) ---
# sig = perceive_v2("I want to cancel my subscription")
# assert sig["intent"] == "cancel"
# assert semantic_search(sig["keywords"], mem.external), "cancel doc should be retrievable"
# print("Task A passed.")

<details>
<summary>Solution — Task A</summary>

```python
def perceive_v2(raw_text: str) -> dict:
    sig = perceive(raw_text)
    if any(w in raw_text.lower() for w in ("cancel", "unsubscribe")):
        sig["intent"] = "cancel"
    return sig

mem.external["cancel_faq"] = ("To cancel, open Settings > Billing and choose "
                              "'Cancel plan'. Access continues until the period ends.")

sig = perceive_v2("I want to cancel my subscription")
assert sig["intent"] == "cancel"
assert semantic_search(sig["keywords"], mem.external)
print("Task A passed.")
```
</details>

**Task B — make escalation sticky.** Once a user has escalated, they should *stay* with a
human for the rest of the session. Add an `escalated` flag to long-term memory, set it in
`store` (or a new step), and make `reason` return `escalate` whenever the flag is set.

In [ ]:
# TODO(you): add a persistent "escalated" flag so escalation sticks within a session.

def reason_v2(signal: dict, ctx: dict) -> dict:
    # TODO(you): if ctx["user"].get("escalated"): return {"action": "escalate", ...}
    return reason(signal, ctx)

# --- check (uncomment once implemented) ---
# mem.long_term["u_42"]["escalated"] = True
# ctx = recall(perceive("thanks, quick question"), mem, "u_42")
# assert reason_v2(perceive("thanks, quick question"), ctx)["action"] == "escalate"
# print("Task B passed.")

<details>
<summary>Solution — Task B</summary>

```python
def reason_v2(signal: dict, ctx: dict) -> dict:
    if ctx["user"].get("escalated"):
        return {"action": "escalate", "reason": "already with a human"}
    plan = reason(signal, ctx)
    if plan["action"] == "escalate":
        ctx["user"]["escalated"] = True   # write the sticky flag to long-term memory
    return plan

mem.long_term["u_42"]["escalated"] = True
ctx = recall(perceive("thanks, quick question"), mem, "u_42")
assert reason_v2(perceive("thanks, quick question"), ctx)["action"] == "escalate"
print("Task B passed.")
```
</details>

## 9 · Key takeaways

- An agent turn is **Perceive → Recall → Reason → Act → Store**, and the loop repeats as
  new input arrives.
- **Memory is read on Recall and written on Store** — never in the reasoning step itself,
  which only *conditions* on what was recalled.
- The three layers are genuinely different objects: short-term is cleared per session,
  long-term persists in a store, external is a retrieved-on-demand knowledge base.
- Delete the Store step and the agent stops learning across turns — repetition and
  missed escalations follow directly.

**Next:** [Agent Architecture & Memory](https://ml-viz-ruby.vercel.app/wiki/agent-architecture-and-memory)
· [The Agentic Project Loop](https://ml-viz-ruby.vercel.app/wiki/agentic-project-loop)